# 각 그룹 별 스피어만 상관계수 확인

In [ ]:
import warnings
import logging
import json
import os
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import ks_2samp

# 1. 환경 설정
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib').setLevel(logging.ERROR)

# ---------------------------------------------------------
# [Config] 경로 및 설정값
# ---------------------------------------------------------
PATHS = {
    "feature_groups": r'..\config\feature_groups.json',
    "remove_lists": [
        # KS < 0.01 + domain knowledge
        r'..\config\remove_features_v1.json',
        r'..\config\remove_features_v2.json',
        
    ],
    "data": r'..\data\fs_data\fs_train.parquet'
}

TARGET_COL = "failure"
CORR_THRESHOLD = 0.90

# ---------------------------------------------------------
# [Functions]
# ---------------------------------------------------------
def load_json(path):
    if not os.path.exists(path): return {}
    with open(path, 'r', encoding='utf-8') as f: return json.load(f)

def get_exclusion_list(paths):
    exclude_set = set()
    for path in paths:
        cfg = load_json(path)
        for key in cfg:
            if isinstance(cfg[key], list): exclude_set.update(cfg[key])
    return exclude_set

def get_ks_score(df, feature_name, target_col):
    fail_vals = df.loc[df[target_col] == 1, feature_name].dropna()
    nonfail_vals = df.loc[df[target_col] == 0, feature_name].dropna()
    if len(fail_vals) == 0 or len(nonfail_vals) == 0: return np.nan, np.nan
    return ks_2samp(fail_vals, nonfail_vals)

# ---------------------------------------------------------
# [Main Logic]
# ---------------------------------------------------------

# 1. 데이터 및 설정 로드
feature_groups = load_json(PATHS["feature_groups"])
exclude_features = get_exclusion_list(PATHS["remove_lists"])
con = duckdb.connect()
df = con.execute(f"SELECT * FROM read_parquet('{PATHS['data']}')").df()

print(f"✅ 분석 준비 완료 (제외 리스트 반영됨)")

# 2. 그룹별 분석 수행
for group_name, cols in feature_groups.items():
    
    # 변수 필터링
    valid_cols = [c for c in cols if (c in df.columns) and (c not in exclude_features)]
    dropped_cols = [c for c in cols if (c in exclude_features) or (c not in df.columns)]
    
    num_total = len(cols)
    num_excluded = len(dropped_cols)
    num_remaining = len(valid_cols)

    # -----------------------------------------------------
    # [출력 형식 변경] 그룹 정보 먼저 출력
    # -----------------------------------------------------
    print(f"\n" + "=" * 60)
    print(f"📂 그룹명: {group_name}")
    print(f"   - 제외된 변수: {num_excluded}개")
    print(f"   - 남은 변수: {num_remaining}개")
    
    if num_remaining < 1:
        print("   ⚠️ 분석할 변수가 없습니다.")
        continue

    # 데이터 전처리
    sub_df = df[valid_cols + [TARGET_COL]].copy()
    for c in valid_cols:
        sub_df[c] = pd.to_numeric(sub_df[c], errors='coerce').astype(float)

    # 상수 컬럼 제외
    feature_only = sub_df[valid_cols]
    feature_only = feature_only.loc[:, feature_only.nunique() > 1]
    
    if num_remaining >= 2 and not feature_only.empty:
        # 상관계수 계산
        feature_only = feature_only.fillna(feature_only.median())
        corr_matrix = feature_only.corr(method='spearman')

        # Heatmap 출력
        n = len(feature_only.columns)
        figsize = (max(7, min(18, n * 0.7)), max(5, min(14, n * 0.5)))
        plt.figure(figsize=figsize)
        sns.heatmap(corr_matrix, annot=(n <= 15), fmt=".2f", cmap='RdBu_r', vmin=-1, vmax=1, center=0)
        plt.title(f"Correlation: {group_name}")
        plt.show()

        # 고중복 탐지
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        redundant = upper.stack().loc[lambda x: abs(x) > CORR_THRESHOLD].sort_values(ascending=False)

        if not redundant.empty:
            print(f"   [!] 고중복 탐지 (|rho| > {CORR_THRESHOLD})")
            for (feat_a, feat_b), corr_val in redundant.items():
                ks_a, p_a = get_ks_score(sub_df, feat_a, TARGET_COL)
                ks_b, p_b = get_ks_score(sub_df, feat_b, TARGET_COL)
                print(f"     • {feat_a} <-> {feat_b} (rho={corr_val:.4f})")
                print(f"       └ KS Score: {feat_a}({ks_a:.4f}), {feat_b}({ks_b:.4f})")
    elif num_remaining == 1:
        print("   ℹ️ 변수가 1개뿐이라 상관행렬을 생성하지 않습니다.")
    else:
        print("   ℹ️ 모든 변수가 상수(constant)이거나 유효하지 않아 분석을 건너뜁니다.")


KeyboardInterrupt: 

In [15]:
import warnings
import logging
import json
import os
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. 초기 설정
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib').setLevel(logging.ERROR)

# 2. 경로 설정 및 JSON 로드
json_path = r'..\data\feature_groups.json'
with open(json_path, 'r', encoding='utf-8') as f:
    feature_groups = json.load(f)

# 3. 데이터 로드
path = r'..\data\fs_data\fs_train.parquet'
con = duckdb.connect()
df = con.execute(f"SELECT * FROM read_parquet('{path}')").df()

# ---------------------------------------------------------
# [추적용 변수 초기화]
# ---------------------------------------------------------
all_json_features = set()
for cols in feature_groups.values():
    all_json_features.update(cols)

calculated_features = set()  # 분석에 실제 사용된 변수
skipped_from_json = set()    # JSON에는 있지만 분석에서 빠진 변수 (결측/상수 등)
missing_in_dataset = set()   # JSON에는 있지만 데이터셋에 없는 변수

# ---------------------------------------------------------
# 그룹별 분석 루프
# ---------------------------------------------------------
threshold = 0.90 

for group_name, cols in feature_groups.items():
    # 1. 존재 여부 체크
    valid_cols = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    missing_in_dataset.update(missing)
    
    if len(valid_cols) < 2:
        skipped_from_json.update(valid_cols)
        continue
    
    # 2. 수치형 변환 및 float 형변환 (Int64 에러 방지)
    sub_df = df[valid_cols].apply(pd.to_numeric, errors='coerce').astype(float)
    
    # 3. 상수 컬럼 제거
    final_cols = sub_df.columns[sub_df.nunique() > 1].tolist()
    const_cols = [c for c in valid_cols if c not in final_cols]
    
    skipped_from_json.update(const_cols)
    calculated_features.update(final_cols)
    
    # 4. 분석 진행 (필터링된 데이터로)
    sub_df = sub_df[final_cols].fillna(sub_df[final_cols].median())
    if sub_df.empty or len(sub_df.columns) < 2: continue

    # Spearman 상관계수 및 Heatmap (기존 시각화 코드 생략 - 유지하면 됨)
    # ... (기존 plt.show() 부분) ...

# ---------------------------------------------------------
# [마지막 요약 출력부]
# ---------------------------------------------------------
# 데이터셋에는 있지만 JSON 그룹화에는 포함되지 않은 변수들
unused_in_dataset = set(df.columns) - all_json_features

print("\n" + "="*60)
print("📌 최종 분석 요약 보고서")
print("="*60)
print(f"✅ 1. 계산된(분석 완료) 변수 개수: {len(calculated_features)}개")
print(f"❌ 2. 계산되지 않은(JSON 내 제외) 변수 개수: {len(skipped_from_json) + len(missing_in_dataset)}개")
print(f"   - 사유(상수/데이터 부족): {len(skipped_from_json)}개")
print(f"   - 사유(데이터셋에 없음): {len(missing_in_dataset)}개")

print("-" * 60)
if missing_in_dataset:
    print(f"⚠️ 3. JSON에는 있으나 데이터셋에 없는 변수 목록 ({len(missing_in_dataset)}개):")
    print(f"   {sorted(list(missing_in_dataset))}")
else:
    print("✅ 3. JSON의 모든 변수가 데이터셋에 존재합니다.")

print("-" * 60)
if unused_in_dataset:
    print(f"ℹ️ 4. 데이터셋에는 있으나 JSON 그룹에 정의되지 않은 변수 목록 ({len(unused_in_dataset)}개):")
    print(f"   {sorted(list(unused_in_dataset))}")
else:
    print("✅ 4. 데이터셋의 모든 변수가 JSON에 정의되어 있습니다.")
print("="*60)


📌 최종 분석 요약 보고서
✅ 1. 계산된(분석 완료) 변수 개수: 304개
❌ 2. 계산되지 않은(JSON 내 제외) 변수 개수: 3개
   - 사유(상수/데이터 부족): 3개
   - 사유(데이터셋에 없음): 0개
------------------------------------------------------------
✅ 3. JSON의 모든 변수가 데이터셋에 존재합니다.
------------------------------------------------------------
ℹ️ 4. 데이터셋에는 있으나 JSON 그룹에 정의되지 않은 변수 목록 (3개):
   ['date', 'failure', 'serial_number']
